# Notebook 04: Advanced ReID — TransReID (Stretch Goal)
**Multi-Camera Tracking System — Kaggle Training Pipeline**

Train TransReID (ViT-Small) for person and vehicle re-identification.
This is a **stretch goal** — only run if baseline models from Notebooks 02-03 are working.

**Model**: ViT-Small/16 with Side Information Embedding (SIE) and Jigsaw Patch Module (JPM)  
**Runtime**: GPU T4/P100 | **Duration**: ~6-8 hours total

## Why TransReID?
- Vision Transformers capture global dependencies vs. CNNs' local receptive fields
- Side Information Embedding encodes camera/viewpoint as learnable tokens
- Jigsaw Patch Module improves robustness to occlusion and misalignment
- State-of-the-art results on Market-1501 and VeRi-776

## Target Metrics
| Dataset | mAP | Rank-1 |
|---------|-----|--------|
| Market-1501 | ≥87% | ≥95% |
| VeRi-776 | ≥80% | ≥96% |

In [ ]:
!pip install -q timm einops torchreid gdown matplotlib pandas

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import math
import shutil
import copy
from pathlib import Path
from datetime import datetime
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import matplotlib.pyplot as plt

try:
    import timm
    print(f"timm: {timm.__version__}")
except ImportError:
    print("Installing timm...")
    os.system("pip install -q timm")
    import timm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/transreid_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. TransReID Model Architecture

We implement a simplified TransReID based on the paper:
> "TransReID: Transformer-based Object Re-Identification" (He et al., ICCV 2021)

Key components:
1. **ViT Backbone** (vit_small_patch16_224 from timm)
2. **Side Information Embedding (SIE)** — learnable camera/viewpoint tokens
3. **Jigsaw Patch Module (JPM)** — patch shuffling for robustness
4. **BNNeck** — batch normalization bottleneck before classification

In [ ]:
class TransReID(nn.Module):
    """Simplified TransReID with ViT-Small backbone."""

    def __init__(
        self,
        num_classes: int,
        num_cameras: int = 0,
        embedding_dim: int = 512,
        vit_model: str = "vit_small_patch16_224",
        pretrained: bool = True,
        sie_camera: bool = True,
        jpm: bool = True,
        drop_rate: float = 0.0,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm

        # ViT backbone from timm
        self.backbone = timm.create_model(
            vit_model,
            pretrained=pretrained,
            num_classes=0,  # Remove classification head
            drop_rate=drop_rate,
        )
        self.feat_dim = self.backbone.embed_dim  # 384 for vit_small

        # Side Information Embedding (camera-aware)
        if self.sie_camera:
            self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, self.feat_dim))
            nn.init.trunc_normal_(self.sie_embed, std=0.02)

        # Global feature branch
        self.bottleneck = nn.BatchNorm1d(self.feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        self.fc = nn.Linear(self.feat_dim, embedding_dim, bias=False)
        self.classifier = nn.Linear(embedding_dim, num_classes, bias=False)

        # Jigsaw Patch Module branch (auxiliary)
        if self.jpm:
            self.jpm_bottleneck = nn.BatchNorm1d(self.feat_dim)
            self.jpm_bottleneck.bias.requires_grad_(False)
            self.jpm_classifier = nn.Linear(self.feat_dim, num_classes, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_normal_(self.fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)
        if self.jpm:
            nn.init.normal_(self.jpm_classifier.weight, std=0.001)

    def forward(self, x, camera_ids=None):
        B = x.shape[0]

        # Patch embedding + position embedding (handled by timm)
        x = self.backbone.patch_embed(x)
        cls_token = self.backbone.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = x + self.backbone.pos_embed

        # Add camera-aware side information
        if self.sie_camera and camera_ids is not None:
            sie = self.sie_embed[camera_ids]  # (B, 1, feat_dim)
            x[:, 0:1, :] = x[:, 0:1, :] + sie

        x = self.backbone.pos_drop(x)

        # Transformer blocks
        for blk in self.backbone.blocks:
            x = blk(x)
        x = self.backbone.norm(x)

        # Global feature from CLS token
        global_feat = x[:, 0]  # (B, feat_dim)

        # BNNeck + projection
        bn_feat = self.bottleneck(global_feat)
        proj_feat = self.fc(bn_feat)  # (B, embedding_dim)

        if self.training:
            cls_score = self.classifier(proj_feat)

            if self.jpm:
                # Jigsaw: shuffle patch tokens and average
                patch_tokens = x[:, 1:]  # Remove CLS
                B, N, C = patch_tokens.shape
                shuffle_idx = torch.randperm(N, device=x.device)
                shuffled = patch_tokens[:, shuffle_idx]
                # Split into 2 groups and average each
                mid = N // 2
                jpm_feat1 = shuffled[:, :mid].mean(dim=1)
                jpm_feat2 = shuffled[:, mid:].mean(dim=1)
                jpm_feat = (jpm_feat1 + jpm_feat2) / 2
                jpm_bn = self.jpm_bottleneck(jpm_feat)
                jpm_score = self.jpm_classifier(jpm_bn)
                return cls_score, proj_feat, jpm_score

            return cls_score, proj_feat

        # Inference: return L2-normalized embedding
        return F.normalize(proj_feat, p=2, dim=1)


def build_transreid(num_classes, num_cameras=0, embedding_dim=512, pretrained=True):
    """Factory function to build TransReID model."""
    model = TransReID(
        num_classes=num_classes,
        num_cameras=num_cameras,
        embedding_dim=embedding_dim,
        pretrained=pretrained,
        sie_camera=num_cameras > 0,
        jpm=True,
    )
    return model

print("TransReID model defined.")
print(f"ViT-Small embed_dim: 384, output embedding_dim: 512")

## 3. Loss Functions

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    """Cross entropy with label smoothing."""

    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_onehot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1 - self.epsilon) * targets_onehot + self.epsilon / self.num_classes
        loss = (-targets_smooth * log_probs).mean(0).sum()
        return loss


class TripletLoss(nn.Module):
    """Triplet loss with hard mining."""

    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        dist = torch.cdist(inputs, inputs, p=2)

        # Hard positive: max distance within same identity
        # Hard negative: min distance across different identities
        mask_pos = targets.unsqueeze(0) == targets.unsqueeze(1)
        mask_neg = ~mask_pos

        # For each anchor, find hardest positive and negative
        dist_ap = (dist * mask_pos.float()).max(dim=1)[0]
        dist_an = (dist + mask_pos.float() * 1e9).min(dim=1)[0]

        y = torch.ones_like(dist_ap)
        loss = self.ranking_loss(dist_an, dist_ap, y)
        return loss


print("Loss functions defined: CrossEntropyLabelSmooth, TripletLoss")

## 4. Training Utilities

In [ ]:
import torchreid


def train_transreid(
    model, datamanager, num_epochs=120, lr=0.008, warmup_epochs=10,
    save_dir="transreid_output", weight_decay=1e-4,
    triplet_weight=1.0, ce_weight=1.0, jpm_weight=0.5
):
    """Train TransReID model with combined losses."""
    model = model.to(DEVICE)

    num_classes = datamanager.num_train_pids
    ce_loss = CrossEntropyLabelSmooth(num_classes).to(DEVICE)
    tri_loss = TripletLoss(margin=0.3).to(DEVICE)

    # Separate lr for backbone (pretrained) vs head (random init)
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = Adam([
        {"params": backbone_params, "lr": lr * 0.1},
        {"params": head_params, "lr": lr},
    ], weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    best_mAP = 0
    train_loader = datamanager.train_loader
    history = {"loss": [], "mAP": [], "rank1": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        num_batches = 0

        # Warmup: linearly increase lr for first N epochs
        if epoch < warmup_epochs:
            warmup_factor = (epoch + 1) / warmup_epochs
            for pg in optimizer.param_groups:
                pg["lr"] = pg["lr"] * warmup_factor / max(warmup_factor, 1e-8)

        for batch_idx, data in enumerate(train_loader):
            imgs = data["img"].to(DEVICE)
            pids = data["pid"].to(DEVICE)
            camids = data.get("camid", None)
            if camids is not None:
                camids = camids.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(imgs, camera_ids=camids)

            if isinstance(outputs, tuple) and len(outputs) == 3:
                cls_score, feat, jpm_score = outputs
                loss = (
                    ce_weight * ce_loss(cls_score, pids)
                    + triplet_weight * tri_loss(feat, pids)
                    + jpm_weight * ce_loss(jpm_score, pids)
                )
            elif isinstance(outputs, tuple) and len(outputs) == 2:
                cls_score, feat = outputs
                loss = ce_weight * ce_loss(cls_score, pids) + triplet_weight * tri_loss(feat, pids)
            else:
                raise ValueError("Unexpected model output format")

            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            num_batches += 1

        scheduler.step()
        avg_loss = running_loss / max(num_batches, 1)
        history["loss"].append(avg_loss)

        # Evaluate periodically
        if (epoch + 1) % 20 == 0 or epoch == num_epochs - 1:
            mAP, rank1 = evaluate_transreid(model, datamanager)
            history["mAP"].append(mAP)
            history["rank1"].append(rank1)

            print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} | mAP: {mAP:.2%} | Rank-1: {rank1:.2%}")

            if mAP > best_mAP:
                best_mAP = mAP
                torch.save(model.state_dict(), save_path / "best_model.pth")
                print(f"  -> New best mAP: {mAP:.2%}")
        else:
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

    # Save final model
    torch.save(model.state_dict(), save_path / "final_model.pth")

    # Save training history
    with open(save_path / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    return history, best_mAP


@torch.no_grad()
def evaluate_transreid(model, datamanager):
    """Evaluate TransReID model on query/gallery."""
    model.eval()

    def extract_features(loader):
        feats, pids, camids = [], [], []
        for data in loader:
            imgs = data["img"].to(DEVICE)
            outputs = model(imgs)
            feats.append(outputs.cpu())
            pids.extend(data["pid"].tolist())
            camids.extend(data.get("camid", [0] * len(data["pid"])))
        return torch.cat(feats), np.array(pids), np.array(camids)

    qf, q_pids, q_camids = extract_features(datamanager.test_loader["query"])
    gf, g_pids, g_camids = extract_features(datamanager.test_loader["gallery"])

    # Compute distance matrix
    dist = torch.cdist(qf, gf, p=2).numpy()

    # Compute CMC and mAP
    num_q = len(q_pids)
    all_AP = []
    all_cmc = np.zeros(len(g_pids))

    for i in range(num_q):
        order = np.argsort(dist[i])
        matches = (g_pids[order] == q_pids[i]).astype(np.int32)
        # Remove same camera same identity
        remove = (g_pids[order] == q_pids[i]) & (g_camids[order] == q_camids[i])
        keep = ~remove
        matches = matches[keep]

        if matches.sum() == 0:
            continue

        cmc = matches.cumsum()
        cmc[cmc > 1] = 1
        all_cmc += cmc[:len(all_cmc)]

        num_rel = matches.sum()
        tmp_cmc = matches.cumsum()
        precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1)
        AP = (precision * matches).sum() / num_rel
        all_AP.append(AP)

    mAP = np.mean(all_AP) if all_AP else 0
    cmc = all_cmc / num_q
    rank1 = cmc[0] if len(cmc) > 0 else 0

    model.train()
    return mAP, rank1


print("Training utilities defined.")

## 5. Train on Market-1501 (Person ReID)

Training TransReID on Market-1501 for person re-identification.  
Expected ~3-4 hours on T4 GPU.

In [ ]:
# Market-1501 datamanager
person_dm = torchreid.data.ImageDataManager(
    root="/kaggle/input/market-1501",
    sources=["market1501"],
    targets=["market1501"],
    height=224,
    width=224,  # ViT expects square input
    batch_size_train=48,  # Smaller batch for ViT memory
    batch_size_test=128,
    transforms=["random_flip", "random_crop", "random_erasing", "color_jitter"],
    num_workers=2,
)

person_model = build_transreid(
    num_classes=person_dm.num_train_pids,
    num_cameras=person_dm.num_train_cams,
    embedding_dim=512,
    pretrained=True,
)

total_params = sum(p.numel() for p in person_model.parameters())
print(f"TransReID parameters: {total_params:,}")
print(f"Train IDs: {person_dm.num_train_pids}, Cameras: {person_dm.num_train_cams}")

In [ ]:
print("=" * 60)
print("Training TransReID on Market-1501 (Person)")
print("=" * 60)

start_time = time.time()
person_history, person_best_mAP = train_transreid(
    person_model, person_dm,
    num_epochs=120, lr=0.008, warmup_epochs=10,
    save_dir=str(OUTPUT_DIR / "person_transreid"),
)
person_time = time.time() - start_time
print(f"\nPerson training completed in {person_time / 3600:.1f} hours")
print(f"Best mAP: {person_best_mAP:.2%}")

## 6. Train on VeRi-776 (Vehicle ReID)

Training TransReID on VeRi-776 for vehicle re-identification.  
Expected ~2-3 hours on T4 GPU.

In [ ]:
vehicle_dm = torchreid.data.ImageDataManager(
    root="/kaggle/input/veri-776",
    sources=["veri"],
    targets=["veri"],
    height=224,
    width=224,
    batch_size_train=48,
    batch_size_test=128,
    transforms=["random_flip", "random_crop", "random_erasing", "color_jitter"],
    num_workers=2,
)

vehicle_model = build_transreid(
    num_classes=vehicle_dm.num_train_pids,
    num_cameras=vehicle_dm.num_train_cams,
    embedding_dim=512,
    pretrained=True,
)

print(f"Train IDs: {vehicle_dm.num_train_pids}, Cameras: {vehicle_dm.num_train_cams}")

In [ ]:
print("=" * 60)
print("Training TransReID on VeRi-776 (Vehicle)")
print("=" * 60)

start_time = time.time()
vehicle_history, vehicle_best_mAP = train_transreid(
    vehicle_model, vehicle_dm,
    num_epochs=120, lr=0.008, warmup_epochs=10,
    save_dir=str(OUTPUT_DIR / "vehicle_transreid"),
)
vehicle_time = time.time() - start_time
print(f"\nVehicle training completed in {vehicle_time / 3600:.1f} hours")
print(f"Best mAP: {vehicle_best_mAP:.2%}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TransReID Training Progress", fontsize=14, fontweight="bold")

for ax, (name, history) in zip(axes, [("Person (Market-1501)", person_history), ("Vehicle (VeRi-776)", vehicle_history)]):
    epochs_loss = range(1, len(history["loss"]) + 1)
    ax.plot(epochs_loss, history["loss"], "b-", alpha=0.7, label="Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color="b")
    ax.tick_params(axis="y", labelcolor="b")
    ax.set_title(name)

    if history["mAP"]:
        ax2 = ax.twinx()
        eval_epochs = [20 * (i + 1) for i in range(len(history["mAP"]))]
        ax2.plot(eval_epochs, [m * 100 for m in history["mAP"]], "r-o", label="mAP")
        ax2.plot(eval_epochs, [r * 100 for r in history["rank1"]], "g-s", label="Rank-1")
        ax2.set_ylabel("Accuracy (%)")
        ax2.legend(loc="lower right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "transreid_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export Models

In [ ]:
export_dir = Path("/kaggle/working/exported_models")
export_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "person_transreid_market1501.pth": OUTPUT_DIR / "person_transreid" / "best_model.pth",
    "vehicle_transreid_veri776.pth": OUTPUT_DIR / "vehicle_transreid" / "best_model.pth",
}

for name, src in exports.items():
    if src.exists():
        dst = export_dir / name
        shutil.copy2(src, dst)
        print(f"Exported: {name} ({dst.stat().st_size / 1024**2:.1f} MB)")
    else:
        print(f"Warning: {src} not found")

metadata = {
    "task": "transreid",
    "models": {
        "person_transreid": {
            "architecture": "transreid_vit_small",
            "dataset": "market1501",
            "embedding_dim": 512,
            "input_size": [224, 224],
            "best_mAP": float(person_best_mAP),
            "file": "person_transreid_market1501.pth",
        },
        "vehicle_transreid": {
            "architecture": "transreid_vit_small",
            "dataset": "veri776",
            "embedding_dim": 512,
            "input_size": [224, 224],
            "best_mAP": float(vehicle_best_mAP),
            "file": "vehicle_transreid_veri776.pth",
        },
    },
    "trained_at": datetime.now().isoformat(),
}

with open(export_dir / "transreid_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nAll TransReID models saved to: {export_dir}")

## 9. Results Summary

In [ ]:
print("=" * 60)
print("TransReID Training Complete")
print("=" * 60)
print(f"\nPerson ReID (Market-1501):")
print(f"  Best mAP: {person_best_mAP:.2%}")
print(f"  Training time: {person_time / 3600:.1f} hours")
print(f"\nVehicle ReID (VeRi-776):")
print(f"  Best mAP: {vehicle_best_mAP:.2%}")
print(f"  Training time: {vehicle_time / 3600:.1f} hours")
print(f"\nTotal GPU time: {(person_time + vehicle_time) / 3600:.1f} hours")
print("\nTo use TransReID locally, you'll need to:")
print("1. Copy the TransReID class definition to src/stage2_features/transreid_model.py")
print("2. Load weights: model.load_state_dict(torch.load('path/to/model.pth'))")
print("3. Update configs/default.yaml: stage2.reid.model_name = 'transreid'")

## Notes for Local Integration

To use TransReID in the pipeline:

1. Copy the `TransReID` class from this notebook to `src/stage2_features/transreid_model.py`
2. Download model weights to `models/reid/`
3. Modify `src/stage2_features/reid_model.py` to support loading TransReID:

```python
if model_name == "transreid":
    from .transreid_model import build_transreid
    model = build_transreid(num_classes=num_classes, num_cameras=num_cameras)
    model.load_state_dict(torch.load(weights_path, map_location=device))
```

4. Update config: set `stage2.reid.model_name: transreid`

### Comparison with Baseline Models
| Model | Params | Dim | Inference Speed | mAP (expected) |
|-------|--------|-----|-----------------|----------------|
| OSNet-x1.0 | ~2.2M | 512 | Fast | ~85% |
| ResNet50-IBN-a | ~25M | 2048 | Medium | ~85% |
| TransReID (ViT-S) | ~22M | 512 | Slower | ~87% |